# Préparation des données SINERT — phase 1 (Kaggle)

Dans Kaggle, ajoute comme **Input** le dataset SINERT (archive ZIP ou dossier de CSV/XLSX). Ajoute aussi `pipeline.py` comme Input (par exemple dans un petit dataset Kaggle privé contenant ce fichier). Les Inputs Kaggle sont en lecture seule; les résultats seront écrits dans `/kaggle/working/phase1`.

Le pipeline lit directement les CSV/XLSX d'un ZIP sans extraire l'archive, ou parcourt un dossier déjà extrait. Si plusieurs archives sont trouvées, choisis explicitement celle à traiter dans la cellule de configuration.

In [ ]:
from pathlib import Path
import os

KAGGLE_INPUT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/phase1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not KAGGLE_INPUT.exists():
    raise RuntimeError('Cette cellule doit être exécutée dans un notebook Kaggle.')

archives = sorted(KAGGLE_INPUT.rglob('*.zip'))
exports = sorted(path for path in KAGGLE_INPUT.rglob('*')
                 if path.is_file() and path.suffix.casefold() in {'.csv', '.xlsx'})
pipeline_files = sorted(KAGGLE_INPUT.rglob('pipeline.py'))
print('Archives ZIP trouvées :')
for path in archives:
    print(f'  {path}')
print('\npipeline.py trouvé :')
for path in pipeline_files:
    print(f'  {path}')

if not pipeline_files:
    raise FileNotFoundError(
        'Ajoute pipeline.py comme Input Kaggle (dataset de code), puis relance.'
    )
PIPELINE = pipeline_files[0]
if len(archives) > 1:
    raise RuntimeError('Plusieurs ZIP trouvés : sélectionne explicitement celui voulu.')
if archives:
    DATA_SOURCE = archives[0]
elif exports:
    # Trouve le dossier commun des exports : cela fonctionne aussi avec un dataset déjà extrait.
    DATA_SOURCE = Path(os.path.commonpath([str(path.parent) for path in exports]))
else:
    raise FileNotFoundError('Aucun ZIP, CSV ou XLSX trouvé sous /kaggle/input.')

print('\nPipeline :', PIPELINE)
print('Données :', DATA_SOURCE)
print('Sortie :', OUTPUT_DIR)

## Dépendances

Kaggle fournit souvent pandas et numpy préinstallés. Cette cellule n'installe que les modules manquants nécessaires aux exports XLSX et Parquet.

In [ ]:
import importlib.util
import subprocess
import sys

required = {'openpyxl': 'openpyxl', 'pyarrow': 'pyarrow<19'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
else:
    print('Dépendances déjà disponibles.')

## Paramètres et exécution

L'horizon ci-dessous est d'un pas (15 minutes), à confirmer avec les encadrantes. Le jeu fériés peut être ajouté plus tard avec `--holidays`.

In [ ]:
import subprocess
import sys

command = [
    sys.executable, str(PIPELINE),
    '--input', str(DATA_SOURCE),
    '--output', str(OUTPUT_DIR),
    '--meter', 'EM_VIRTUAL',
    '--window-steps', '672',
    '--horizon-steps', '1',
    '--max-short-gap', '4',
]
print('Commande :', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

report_path = OUTPUT_DIR / 'quality_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report, ensure_ascii=False, indent=2))

clean = pd.read_parquet(OUTPUT_DIR / 'sinert_clean.parquet')
windows = pd.read_parquet(OUTPUT_DIR / 'window_index.parquet')
print('\nDonnées nettoyées :', clean.shape)
print('Fenêtres :', windows.shape)

physical_columns = report['target_columns']
missing_summary = pd.DataFrame({
    column: {
        'valeurs_manquantes': details['missing_values'],
        'pourcentage': details['missing_percent'],
        'premiere_valeur_valide': details['first_valid_timestamp'],
        'derniere_valeur_valide': details['last_valid_timestamp'],
    }
    for column, details in report['physical_quality_after_imputation'].items()
}).T
print('\nQualité des mesures après imputation :')
display(missing_summary)
print('Lignes comportant au moins une mesure encore manquante :',
      report['rows_with_unresolved_physical_missing'])
print('Lignes où au moins une valeur a été imputée :',
      report['rows_with_any_imputed_physical_value'])
print('\nPlus longues lacunes restantes par variable :')
for column, details in report['physical_quality_after_imputation'].items():
    print(f"\n{column}:")
    display(pd.DataFrame(details['longest_missing_runs']))
print('\nRemarque : une ligne peut être non imputée et rester manquante. '
      'was_imputed indique une valeur synthétique; '
      'has_missing_physical indique une mesure toujours absente.')
display(clean.head())
display(windows.groupby('split').size().rename('nombre_de_fenetres'))

import matplotlib.pyplot as plt

if clean['was_imputed'].any():
    first_imputed = clean.index[clean['was_imputed']][0]
    plot_start = first_imputed - pd.Timedelta(hours=12)
    plot_end = first_imputed + pd.Timedelta(hours=36)
    plot_data = clean.loc[plot_start:plot_end]
    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
    for ax, column in zip(axes, physical_columns):
        imputed_column = f'imputed_{column}'
        ax.plot(plot_data.index, plot_data[column], color='steelblue', linewidth=1,
                label='Valeur nettoyée')
        imputed_points = plot_data[imputed_column].fillna(False)
        ax.scatter(plot_data.index[imputed_points], plot_data.loc[imputed_points, column],
                   color='crimson', s=24, label='Valeur imputée', zorder=3)
        ax.set_ylabel(column)
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.25)
    fig.suptitle(f'Exemple de zones imputées : {plot_start} — {plot_end}')
    fig.tight_layout()
    plt.show()
else:
    print('Aucune valeur imputée : rien à tracer.')

print('\nFichiers livrables :')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path, f'({path.stat().st_size:,} octets)')